<a href="https://colab.research.google.com/github/dinanrzki/Junior-Data-Analyst-Project/blob/main/client_hotel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =====================================================================
# BAGIAN A: ANALISIS LOG TRANSAKSI HOTEL (hotel_logs.csv)
# =====================================================================
print("=" * 50)
print("PENGOLAHAN DATA LOG TRANSAKSI (hotel_logs.csv)")
print("=" * 50)

# 1. Load Data & Eksplorasi Awal menggunakan variabel khusus log
df_logs = pd.read_csv("hotel_logs.csv")
df_logs['booking_time'] = pd.to_datetime(df_logs['booking_time'])
df_logs['net_rev'] = df_logs['gross_rev'] * (1 - df_logs['comm_fee_pct'])
df_logs['hour'] = df_logs['booking_time'].dt.hour
print("✅ Data hotel_logs.csv berhasil dimuat.")

# 2. Ambil subset data khusus Direct Web dan transaksi Sukses dari df_logs
dw_logs = df_logs[df_logs['platform'] == 'Direct_Web']
success_logs = df_logs[df_logs['trx_status'] == 'Success']

# 3. Analisis Bukti Masalah Sistemik - Analisis per Jam
hourly_fail = dw_logs.groupby("hour")["trx_status"].apply(
    lambda x: (x=="Failed").sum() / len(x) * 100
).round(1)

print(f"Min failure rate per jam : {hourly_fail.min():.1f}%")
print(f"Max failure rate per jam : {hourly_fail.max():.1f}%")
print(f"Std deviation            : {hourly_fail.std():.1f}%")
print("Kesimpulan: Masalah SISTEMIK, bukan server overload\n")

# 4. Kalkulasi Dampak Finansial (Komisi OTA)
rev = success_logs.groupby("platform").agg(
    gross = ("gross_rev", "sum"),
    net = ("net_rev", "sum"),
    avg_txn = ("gross_rev", "mean"),
).reset_index()

rev['komisi'] = rev['gross'] - rev['net']
total_komisi = rev["komisi"].sum()
print(f"Total komisi ke OTA: Rp {total_komisi/1e6:.1f} Juta")

# 5. Kalkulasi Potensi Revenue Hilang dari CC failure di Direct Web
cc_failed = dw_logs[(dw_logs['pay_method'] == 'Credit Card') & (dw_logs['trx_status'] == 'Failed')]
avg_ticket = success_logs["gross_rev"].mean()
lost_rev = len(cc_failed) * avg_ticket
print(f"Potensi revenue hilang: Rp {lost_rev/1e6:.1f} Juta")


# =====================================================================
# BAGIAN B: DATA CLEANING & ABSA (File Ulasan/Review Hotel)
# =====================================================================
print("\n" + "=" * 50)
print("PROSES DATA CLEANING & ABSA (Review Hotel)")
print("=" * 50)

# Gunakan data dummy untuk review agar tidak menimpa variabel logs kamu
data_review_dummy = {
    'Tanggal': ['2026-06-01'],
    'Platform': ['agoda'],
    'Rating': ['8.5/10'],
    'Review': ['<p>Kamarnya sangat bersih dan pelayanan staf ramah gila!</p>']
}
df_reviews = pd.DataFrame(data_review_dummy)

# 1. Hapus HTML tags
df_reviews['Review'] = df_reviews['Review'].str.replace(r'<[^>]+>', '', regex=True).str.strip()
print("✅ HTML tags dibersihkan")

# 2. Hapus duplikat
before = len(df_reviews)
df_reviews = df_reviews.drop_duplicates(subset=['Review'])
print(f"✅ Duplikat dihapus: {before - len(df_reviews)} baris")

# 3. Standarisasi Platform
def fix_platform(p):
    p = str(p).strip().lower()
    if 'agoda' in p: return 'Agoda'
    if 'traveloka' in p or 'tvlk' in p: return 'Traveloka'
    if 'booking' in p: return 'Booking.com'
    return p.title()

df_reviews['Platform'] = df_reviews['Platform'].apply(fix_platform)
print("✅ Platform distandarisasi")

# 4. Normalisasi Rating ke skala 1-10
def normalize_rating(r):
    r = str(r).strip()
    m = re.match(r'^(\d+\.?\d*)\s*/\s*10$', r)
    if m: return float(m.group(1))
    m = re.match(r'^(\d+\.?\d*)\s*/\s*5$', r)
    if m: return float(m.group(1)) * 2
    m = re.match(r'^(\d+\.?\d*)$', r)
    if m:
        val = float(m.group(1))
        if val <= 10: return val
    return None

df_reviews['Rating_Normalized'] = df_reviews['Rating'].apply(normalize_rating)
df_reviews = df_reviews.dropna(subset=['Rating_Normalized'])
df_reviews['Rating_Normalized'] = df_reviews['Rating_Normalized'].astype(float)
df_reviews = df_reviews.reset_index(drop=True)

print("✅ Rating dinormalisasi ke skala 1-10")
print(f"Rating avg    : {df_reviews['Rating_Normalized'].mean():.2f}/10")
print("✅ Sistem ABSA Siap digunakan.")

PENGOLAHAN DATA LOG TRANSAKSI (hotel_logs.csv)
✅ Data hotel_logs.csv berhasil dimuat.
Min failure rate per jam : 20.0%
Max failure rate per jam : 58.3%
Std deviation            : 11.4%
Kesimpulan: Masalah SISTEMIK, bukan server overload

Total komisi ke OTA: Rp 400.0 Juta
Potensi revenue hilang: Rp 0.0 Juta

PROSES DATA CLEANING & ABSA (Review Hotel)
✅ HTML tags dibersihkan
✅ Duplikat dihapus: 0 baris
✅ Platform distandarisasi
✅ Rating dinormalisasi ke skala 1-10
Rating avg    : 8.50/10
✅ Sistem ABSA Siap digunakan.
